# Simple MLP Version

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Import project modules
from config import STRAINS, ATCC_GROUPINGS, antibiotics, HIERARCHICAL_GROUPINGS, HIERARCHICAL_CLASSES

def set_seed(seed=123):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(123)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = "data"
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Load the dataset
X_ref = np.load(os.path.join(DATA_DIR, "X_reference.npy"))
y_ref = np.load(os.path.join(DATA_DIR, "y_reference.npy")).astype(int)
X_te = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_te = np.load(os.path.join(DATA_DIR, "y_test.npy")).astype(int)

print("Dataset shapes:")
print(f"X_reference: {X_ref.shape}")
print(f"y_reference: {y_ref.shape}")
print(f"X_test: {X_te.shape}")
print(f"y_test: {y_te.shape}")

# Check strain distribution
print(f"\nStrain distribution in reference set:")
strain_counts = Counter(y_ref)
for strain_id, count in sorted(strain_counts.items()):
    print(f"Strain {strain_id} ({STRAINS[strain_id]}): {count} samples")

In [ ]:
def get_hierarchical_strains():
    """Get strain IDs that belong to MSSA or MRSA classes"""
    hierarchical_strains = [14, 15, 16, 17, 18]  # MSSA and MRSA strains
    return sorted(hierarchical_strains)

# Get focused strains for hierarchical classification
focused_strains = get_hierarchical_strains()
print("\nHIERARCHICAL labels: strain -> superclass mapping")

for sid in focused_strains:
    superclass_id = HIERARCHICAL_GROUPINGS[sid]
    strain_name = STRAINS[sid]
    superclass_name = HIERARCHICAL_CLASSES[superclass_id]
    print(f"{sid:2d} | {strain_name:<20} : [{superclass_id}] {superclass_name}")

In [ ]:
STRAIN_ORDER = np.arange(30)  
sid_ref_all = np.repeat(STRAIN_ORDER, 2000)
sid_te_all  = np.repeat(STRAIN_ORDER, 100)

focused_strains = get_hierarchical_strains()
# Create hierarchical class mapping
HIERARCHICAL_TO_BIN = {0: 0, 1: 1}  # MSSA=0, MRSA=1
BIN_TO_HIERARCHICAL = {0: 0, 1: 1}

# Filter data for hierarchical classification
ref_mask = np.isin(sid_ref_all, focused_strains)
te_mask = np.isin(sid_te_all, focused_strains)

sid_ref_filtered = sid_ref_all[ref_mask]
sid_te_filtered = sid_te_all[te_mask]

X_ref_filtered = X_ref[ref_mask]
X_te_filtered = X_te[te_mask]

# Create hierarchical labels
y_ref_filtered = np.array([HIERARCHICAL_TO_BIN[HIERARCHICAL_GROUPINGS[s]] for s in sid_ref_filtered], dtype=int)
y_te_filtered = np.array([HIERARCHICAL_TO_BIN[HIERARCHICAL_GROUPINGS[s]] for s in sid_te_filtered], dtype=int)

print(f"\nHierarchical classification data shapes:")
print(f"X_ref_filtered: {X_ref_filtered.shape}")
print(f"y_ref_filtered: {y_ref_filtered.shape}")
print(f"X_te_filtered: {X_te_filtered.shape}")
print(f"y_te_filtered: {y_te_filtered.shape}")

# Show class distribution
print(f"\nClass distribution in hierarchical dataset:")
from collections import Counter
class_counts = Counter(y_ref_filtered)
for class_id, count in sorted(class_counts.items()):
    class_name = HIERARCHICAL_CLASSES[class_id]
    print(f"Class {class_id} ({class_name}): {count} samples")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BinaryClassifier(nn.Module):
    def __init__(self, input_dim=1000, hidden_dims=[512, 256, 128], dropout=0.1):  # Reduced dropout
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev, h, bias=True),
                nn.BatchNorm1d(h),  # Use BatchNorm instead of LayerNorm
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            prev = h
        layers += [nn.Linear(prev, 2)]
        self.network = nn.Sequential(*layers)
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)  # More stable initialization
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.network(x)

In [ ]:
def prepare_data(X, y, batch_size=32, shuffle=True):
    """Convert numpy arrays to DataLoader with shape [B,L] for the new model."""
    X_tensor = torch.as_tensor(X, dtype=torch.float32)  # [N,L] - no unsqueeze
    y_tensor = torch.as_tensor(y, dtype=torch.long)
    dataset  = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False)

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for data, target in train_loader:
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad(set_to_none=True)
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        # sample-weighted loss to avoid bias from a short last batch
        bs = target.size(0)
        total_loss += loss.detach().item() * bs
        pred = output.detach().argmax(dim=1)
        correct += (pred == target).sum().item()
        total += bs

    avg_loss = total_loss / max(total, 1)
    acc = 100.0 * correct / max(total, 1)
    return avg_loss, acc

In [ ]:
# Evaluation function
def evaluate(model, test_loader, criterion, device):
    model.eval()
    assert not model.training, "Model must be in eval() during evaluation."

    total_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)

            bs = target.size(0)
            total_loss += loss.detach().item() * bs
            pred = output.detach().argmax(dim=1)

            correct += (pred == target).sum().item()
            total += bs
            all_preds.extend(pred.cpu().tolist())
            all_targets.extend(target.cpu().tolist())

    avg_loss = total_loss / max(total, 1)
    acc = 100.0 * correct / max(total, 1)
    return avg_loss, acc, all_preds, all_targets

In [ ]:
def train_eval_permutation_with_epochs(
    X_train, y_train, X_test, y_test,
    epochs=10, lr=1e-4, batch_size=256, weight_decay=1e-3
):
    
    torch.manual_seed(123)
    torch.cuda.manual_seed_all(123)
    
    # Infer spectrum length from data
    input_dim = X_train.shape[1]
    assert X_test.shape[1] == input_dim, "Train/Test spectrum length mismatch."

    # Model (RamanNet) + loaders
    model = BinaryClassifier(input_dim=input_dim).to(DEVICE)
    train_loader = prepare_data(X_train, y_train, batch_size=batch_size, shuffle=True)
    test_loader  = prepare_data(X_test,  y_test,  batch_size=batch_size, shuffle=False)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_test_acc = -1.0
    best_epoch = -1
    best_state = None

    for epoch in range(1, epochs + 1):
        # ---- Train ----
        model.train()
        running_correct, running_total, running_loss = 0, 0, 0.0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)  #  [B,L] 
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.detach().item() * xb.size(0)
            preds = logits.argmax(dim=1)
            running_correct += (preds == yb).sum().item()
            running_total += xb.size(0)

        train_acc = 100.0 * running_correct / max(1, running_total)

        # ---- Eval ----
        model.eval()
        test_correct, test_total, test_loss_sum = 0, 0, 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)

                logits = model(xb)
                loss = criterion(logits, yb)

                test_loss_sum += float(loss) * xb.size(0)
                preds = logits.argmax(dim=1)
                test_correct += (preds == yb).sum().item()
                test_total += xb.size(0)

        test_acc = 100.0 * test_correct / max(1, test_total)
        print(f"    Epoch {epoch}: Train Acc = {train_acc:.2f}%, Test Acc = {test_acc:.2f}%")

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"[BEST] Epoch {best_epoch}: Test Acc = {best_test_acc:.2f}%")
    return best_test_acc / 100.0

In [ ]:
LAST_PERM_MAP = None
from itertools import combinations


def generate_all_unique_permutations(focused_strains):
    """
    Generate all 10 unique ways to assign 3 strains to MSSA and 2 to MRSA
    """
    n_mssa = sum(HIERARCHICAL_GROUPINGS[s] == 0 for s in focused_strains)  # 3
    n_mrsa = sum(HIERARCHICAL_GROUPINGS[s] == 1 for s in focused_strains)  # 2
    
    # Generate all combinations of 3 strains for MSSA
    mssa_combinations = list(combinations(focused_strains, n_mssa))
    
    unique_permutations = []
    for mssa_strains in mssa_combinations:
        mrsa_strains = [s for s in focused_strains if s not in mssa_strains]
        
        # Create strain-to-label mapping
        strain_to_label = {}
        for strain in mssa_strains:
            strain_to_label[strain] = HIERARCHICAL_TO_BIN[0]  # MSSA = 0
        for strain in mrsa_strains:
            strain_to_label[strain] = HIERARCHICAL_TO_BIN[1]  # MRSA = 1
            
        unique_permutations.append(strain_to_label)
    
    return unique_permutations

def print_hierarchical_permutation_mapping(perm_idx, y_ref_perm, y_te_perm):
    print(f"\nPermutation {perm_idx}: strain -> hierarchical class (TRUE -> PERMUTED)")
    if LAST_PERM_MAP is None:
        print("  [warn] LAST_PERM_MAP is not set")
        return
    for sid in focused_strains:
        true_superclass = HIERARCHICAL_GROUPINGS[sid]             
        perm_bin = LAST_PERM_MAP[sid]             
        perm_superclass = BIN_TO_HIERARCHICAL[perm_bin]         
        strain_name = STRAINS[sid]
        true_name = HIERARCHICAL_CLASSES[true_superclass]
        perm_name = HIERARCHICAL_CLASSES[perm_superclass]
        print(f"{sid:2d} | {strain_name:<20} : [{true_superclass}] {true_name}  ->  [{perm_superclass}] {perm_name}")

In [ ]:
PERM_EPOCHS = 8  # Epochs for each permutation
RANDOM_SEED =42

In [ ]:
# Train with true hierarchical labels
print(f"\nTRUE hierarchical labels training:")
true_accuracy = train_eval_permutation_with_epochs(
    X_ref_filtered, y_ref_filtered,
    X_te_filtered, y_te_filtered,
    epochs=PERM_EPOCHS
)
print(f"TRUE hierarchical labels final accuracy: {true_accuracy:.2%}")


In [ ]:

null_accuracies = []


ref_strains_order = sid_ref_filtered   
te_strains_order = sid_te_filtered    

# Generate all unique permutations
unique_permutations = generate_all_unique_permutations(focused_strains)

for perm_idx, strain_to_label in enumerate(unique_permutations, 1):
    # Apply the same mapping to both splits
    y_ref_permuted = np.array([strain_to_label[s] for s in ref_strains_order], dtype=int)
    y_te_permuted = np.array([strain_to_label[s] for s in te_strains_order], dtype=int)
    
    # Print the mapping
    print(f"\nPermutation {perm_idx}: strain -> hierarchical class (TRUE -> PERMUTED)")
    for sid in focused_strains:
        true_superclass = HIERARCHICAL_GROUPINGS[sid]
        perm_bin = strain_to_label[sid]
        perm_superclass = BIN_TO_HIERARCHICAL[perm_bin]
        strain_name = STRAINS[sid]
        true_name = HIERARCHICAL_CLASSES[true_superclass]
        perm_name = HIERARCHICAL_CLASSES[perm_superclass]
        print(f"{sid:2d} | {strain_name:<20} : [{true_superclass}] {true_name}  ->  [{perm_superclass}] {perm_name}")
    
    # Train & evaluate with permuted labels
    print(f"\nPermutation {perm_idx} training:")
    perm_accuracy = train_eval_permutation_with_epochs(
        X_ref_filtered, y_ref_permuted,
        X_te_filtered, y_te_permuted,
        epochs=PERM_EPOCHS
    )
    null_accuracies.append(perm_accuracy)
    print(f"Permutation {perm_idx} final accuracy: {perm_accuracy:.2%}")

print("Unique hierarchical permutation test completed!")

In [ ]:
# Create histogram visualization for hierarchical permutation test
null_accuracies = np.array(null_accuracies)

plt.figure(figsize=(7, 5))
plt.hist(null_accuracies * 100, bins=20, color='navy', alpha=0.7)

# Observed performance line
plt.axvline(true_accuracy * 100, color='red', linewidth=2, label='True Performance')

# Chance baseline for binary classification = 50%
plt.axvline(50, color='k', linestyle=':', linewidth=1, label='Chance (50%)')

plt.xlabel("Accuracy [%] (on the test set)")
plt.ylabel("Frequency")
plt.title("Permutation null vs observed (Hierarchical: MSSA vs MRSA)")

# Add text annotation with observed accuracy and permutation p-value
ymax = max(np.histogram(null_accuracies * 100, bins=20)[0])
p_value = np.mean(null_accuracies >= true_accuracy)

plt.text(true_accuracy * 100 + 2, ymax * 0.6,
         f"Observed = {true_accuracy * 100:.2f}%\nP = {p_value:.4f}",
         fontsize=12, bbox=dict(facecolor='yellow', alpha=0.7))

plt.legend()
plt.grid(True, alpha=0.3)
plt.show()